In [235]:
import numpy as np
import pandas as pd

In [236]:
def split_train_test(x, y, train_size=0.8, random_state=42):
    train_set_size = int(len(x) * train_size)
    
    np.random.seed(random_state)
    shuffled_indices = np.random.permutation(len(x))
        
    train_indices = shuffled_indices[:train_set_size]
    test_indices = shuffled_indices[train_set_size:]
    
    return np.array(x.iloc[train_indices]), np.array(x.iloc[test_indices]), np.array(y.iloc[train_indices]), np.array(y.iloc[test_indices])

In [237]:
cancer_data = pd.read_csv("./input/breast_cancer.csv")
cancer_data.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [238]:
x_train, x_test, y_train, y_test = split_train_test(cancer_data.drop("diagnosis", axis=1), cancer_data["diagnosis"])
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((455, 31), (114, 31), (455,), (114,))

In [239]:
x_train_benign = x_train[y_train == "B"]
x_train_malignant = x_train[y_train == "M"]

beningn_prior = len(x_train_benign) / len(x_train)
malignant_prior = len(x_train_malignant) / len(x_train)

x_train_benign_means = np.mean(x_train_benign, axis=0)
x_train_benign_vars = np.var(x_train_benign, axis=0)

x_train_malignant_means = np.mean(x_train_malignant, axis=0)
x_train_malignant_vars = np.var(x_train_malignant, axis=0)

priors = {
    "B": beningn_prior,
    "M": malignant_prior
}

means = {
    "B": x_train_benign_means,
    "M": x_train_malignant_means
}

variances = {
    "B": x_train_benign_vars,
    "M": x_train_malignant_vars
}

In [240]:
def compute_feature_probability(x, feature_index):
    benign_feature_mean = means['B'][feature_index]
    malignant_feature_mean = means['M'][feature_index]
    
    benign_feature_var = variances['B'][feature_index]
    malignant_feature_var = variances['M'][feature_index]
    
    a1 = 1 / np.sqrt(2 * np.pi * benign_feature_var)
    b1 = (x[feature_index] - benign_feature_mean) ** 2
    c1 = 2 * benign_feature_var
    
    benign_feature_prob = a1 * np.exp(-b1 / c1)
    
    a2 = 1 / np.sqrt(2 * np.pi * malignant_feature_var)
    b2 = (x[feature_index] - malignant_feature_mean) ** 2
    c2 = 2 * malignant_feature_var
    
    malignant_feature_prob = a2 * np.exp(-b2 / c2)
    
    return benign_feature_prob, malignant_feature_prob

In [241]:
def compute_class_probabilities(x):
    benign_score = np.log(priors['B'])
    malignant_score = np.log(priors['M'])
    
    for feature_index in range(len(x)):
        benign_feature_prob, malignant_feature_prob = compute_feature_probability(x, feature_index)
        
        benign_score += np.log(benign_feature_prob)
        malignant_score += np.log(malignant_feature_prob)
    
    return benign_score, malignant_score

In [242]:
def predict(x):
    benign_score, malignant_score = compute_class_probabilities(x)
    return "B" if benign_score > malignant_score else "M"

In [243]:
def test_accuracy(x_test, y_test):
    correct_predictions = 0
    
    for x, y in zip(x_test, y_test):
        prediction = predict(x)
        
        if prediction == y:
            correct_predictions += 1
            
    return correct_predictions / len(x_test)

In [244]:
accuracy = test_accuracy(x_test, y_test)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.93


/tmp/ipykernel_948899/2616101255.py:8: RuntimeWarning: divide by zero encountered in log
  benign_score += np.log(benign_feature_prob)
